# Predicting HOMO/LUMO energies from SMILES with collision-free Morgan fingerprints

**Project report — QM9, CF-MF, SHAP group attribution**

---

## 1. Objective

Two goals, in order of importance:

1. **Interpretability (primary).** Quantify how individual functional groups shift
   frontier orbital energies — magnitude and direction, per group, in eV.
2. **Prediction (supporting).** Build accurate HOMO/LUMO models from SMILES alone,
   good enough that their attributions are worth trusting.

The interpretability goal follows Li et al., *J. Cheminformatics* **18**:46 (2026),
"Collision-free Morgan fingerprints", extended from their per-molecule atom-level
attribution on three example compounds to **dataset-wide group-level aggregation**.

---

## 2. Why collision-free fingerprints

A standard Morgan fingerprint assigns each circular atom environment a 32-bit
identifier, then folds it into a fixed number of bits with `identifier mod L`.
Distinct substructures land on the same bit — a **collision**. The model is then
forced to share one parameter between unrelated chemical motifs, and any SHAP value
on that bit is a blend of several substructures' effects. Interpretation becomes
unreliable.

**Key simplification used here.** The paper's algorithm iteratively grows `L` until no
two identifiers collide, then deletes the empty bits ("compaction"). After compaction
every column corresponds to exactly one identifier — which is the same thing you get
by skipping folding entirely and giving each observed identifier its own column.
So the length search is unnecessary: RDKit's unfolded sparse fingerprint plus a
`np.unique` over the observed identifiers produces an identical feature set with no
iteration.

This also eliminated a bug in the earlier version of this project, where the
collision-free length was determined on a 20k subset and then applied to all 134k
molecules — reintroducing collisions for the 113k molecules never checked.

**Verified:** 180,337 distinct identifiers at radius 2, **zero pre-hash collisions**
(no identifier appears at two different radii).

---

## 3. Pipeline

### 3.1 Data cleaning (`01_audit.py`, `02_clean.py`)

| step | molecules removed | remaining |
|---|---|---|
| raw QM9 | — | 133,885 |
| uncharacterized (failed DFT geometry consistency) | 3,054 | 130,831 |
| fewer than 2 heavy atoms | 3 | 130,828 |
| duplicate canonical SMILES (labels averaged) | 87 | **130,741** |

Audit findings:
- Elements: C, N, O, F only (+ implicit H). No charged or multi-fragment molecules.
- Labels are rounded to 4 decimal places in Hartree → **label resolution ≈ 0.003 eV**.
  This explains the maximum duplicate disagreement of exactly 0.0027 eV.
- `gap = lumo − homo` holds to 1e-4 Ha, consistent with that rounding.

**Baselines (mean predictor):**

| target | std (eV) | MAE (eV) |
|---|---|---|
| HOMO | 0.598 | 0.440 |
| LUMO | 1.274 | 1.051 |

LUMO has 2.1× the spread of HOMO. This matters throughout: **R² flatters LUMO**
because it normalises by target variance. MAE in eV is the honest comparison.

### 3.2 Splits (`03_split.py`)

80 / 10 / 10 → 104,592 / 13,074 / 13,075 molecules, stored by `mol_id`.

- **Random split** — the standard QM9 benchmark.
- **Scaffold split** — Bemis–Murcko ring frameworks kept entirely on one side.
  15,661 unique scaffolds, 12,296 of them singletons. The 13,387 acyclic molecules
  form one group too large for val/test, so they are forced into train: **the
  scaffold test set contains no acyclic molecules**, and its HOMO mean is shifted
  +0.14 eV relative to train.

All hyperparameter tuning was done on the **random-split** validation set and the
chosen parameters reused unchanged for the scaffold models, so the two are
comparable. The scaffold val set is dominated by a few large scaffold families and
would give a noisy tuning signal.

Everything fitted — descriptor filters, fingerprint vocabularies, scalers —
is fitted on training molecules only, separately per split.

### 3.3 Features (`04a_descriptors.py`, `04b_cfmf.py`)

**Descriptors.** All 217 RDKit 2D descriptors, then a train-fitted filter:
drop non-finite → drop constant → drop one of any pair correlating |r| > 0.95.
Result: **162 descriptors** (random split), of which 53 are `fr_*` functional
group counts.

**CF-MF.** Unfolded count-based Morgan fingerprints, radius 2.

| | count |
|---|---|
| distinct identifiers | 180,337 |
| radius 0 (atom types) | 44 |
| radius 1 | 3,916 |
| radius 2 | 176,377 |
| pre-hash collisions | **0** |
| mean identifiers per molecule | 20.5 |

Two train-fitted column sets: `min1` (present in ≥1 training molecule, 161,375
columns) and `min10` (≥10 molecules, 13,749 columns).

Note: **there are more radius-2 identifiers (176k) than molecules (131k)**. With
≤9 heavy atoms, a radius-2 environment covers most of a molecule, so individual
radius-2 columns are nearly molecule-specific. This is why group aggregation is
necessary rather than optional.

### 3.4 Configuration screening (`05a_screen.py`, `05a_followup.py`)

Eight fingerprint configurations compared with Ridge (fast, exact), validation MAE:

| configuration | HOMO | LUMO |
|---|---|---|
| r≤2, counts, min1 | **0.120** | **0.144** |
| r≤2, counts, min10 | 0.140 | 0.200 |
| r≤1, counts, min1 | 0.173 | 0.227 |
| r≤2, binary, min1 | 0.127 | 0.147 |

Conclusions: **radius 2 ≫ radius 1**, **counts > binary** (marginally),
**min1 > min10** for Ridge.

A follow-up on the scaffold split tested whether the min1 advantage was just
near-twin memorisation: it shrank by ~40% but survived, so rare substructures
carry some genuinely transferable signal for an additive model.

### 3.5 Tuning (`05b`, `05c`)

Random search, 20 candidates, random-split val set, separate models per target.

- **RF on descriptors:** `max_features=0.7, min_samples_leaf=1, max_depth=25`.
  The top 5 candidates were within 0.002–0.005 eV — the model is insensitive to
  hyperparameters, which is reassuring for robustness.
- **LightGBM on CF-MF:** `num_leaves=255, min_child_samples=10`. Most top candidates
  hit the 5,000-tree cap, so these val MAEs are slightly pessimistic; the final
  models were refit at learning rate 0.05 with a higher cap.

---

## 4. Prediction results

### 4.1 Test set (touched once)

Test MAE in eV, **random split / scaffold split**:

| model | features | HOMO | LUMO |
|---|---|---|---|
| mean predictor | — | 0.440 | 1.051 |
| RF | 162 descriptors | 0.121 / 0.172 | 0.125 / 0.174 |
| Ridge | CF-MF min1 | 0.118 / 0.158 | 0.143 / 0.192 |
| **LightGBM** | CF-MF min10 | **0.098 / 0.150** | **0.094 / 0.145** |

R² (random split): LightGBM 0.944 (HOMO) / 0.988 (LUMO).
RMSE/MAE ≈ 1.4–1.5 throughout — near-Gaussian errors, no heavy tail.

### 4.2 Two observations

**HOMO and LUMO are equally accurate in eV** (0.098 vs 0.094), despite R² differing
by 0.04. The R² gap is entirely LUMO's larger variance.

**The best model degrades most out of distribution.** Random → scaffold:

| model | HOMO | LUMO |
|---|---|---|
| Ridge | +34% | +34% |
| RF descriptors | +42% | +39% |
| LightGBM | +53% | +54% |

LightGBM extracts the most from the training distribution, including fine-grained
patterns that don't transfer to unseen ring systems. Ridge's simple additive form
has less scaffold-specific structure to lose. The ranking is unchanged on scaffold,
so LightGBM is still best — it just loses more of its advantage.

A contributing mechanism: the scaffold training set contains 151,916 identifiers
vs 161,375 for random, so the scaffold test set is full of substructures the model
has never seen, which contribute nothing.

### 4.3 Literature placement

These models see **only SMILES**. The QM9 leaderboard uses **3D DFT-optimised
coordinates**, which is a substantially easier problem.

| class | HOMO | LUMO |
|---|---|---|
| SOTA 3D GNN (2026) | 0.010 | 0.009 |
| DimeNet++ / ALIGNN | 0.025 / 0.022 | 0.020 |
| SchNet | 0.04 | 0.03 |
| **this work (2D only)** | **0.098** | **0.094** |
| kernel ridge, Coulomb matrix | 0.18 | 0.25 |

The relevant floor is not 0.010 eV. Molecules with identical SMILES but different
3D conformers differ in HOMO by **≈ 0.028 eV** in comparable B3LYP data — a hard
limit for any model that never sees geometry. So this work sits ~3.5× above the
SMILES-only floor, not 10× above an attainable target. Independently, on the
Molecule3D benchmark, dropping from ground-truth 3D to a 2D graph costs a factor
of 3 (0.070 → 0.209 eV on the gap).

### 4.4 Ablations (`14_ablations.py`)

**Which descriptors mattered?** Same data, split and hyperparameters; only columns differ.

| variant | n | HOMO | LUMO |
|---|---|---|---|
| original 20 (15 survived filtering) | 15 | 0.162 | 0.208 |
| full set minus electronic | 122 | 0.124 | 0.138 |
| full set minus `fr_*` | 109 | 0.122 | 0.128 |
| **full set** | 162 | **0.121** | **0.125** |

- Expanding the descriptor set cut MAE by 26% (HOMO) and 40% (LUMO).
- Electronic descriptors help **LUMO 4× more than HOMO** (+0.012 vs +0.003 eV when
  removed) — the opposite of the initial hypothesis.
- No single family explains the gain. Each removal costs ≤0.012 eV while the total
  gain is 0.04–0.08 eV, i.e. the improvement comes from the breadth of the set and
  individual families are redundant with one another.

**Do rare fingerprint columns help LightGBM?**

| | n columns | HOMO | LUMO |
|---|---|---|---|
| min10 | 13,749 | 0.1071 | 0.1054 |
| min1 | 161,375 | 0.1073 | 0.1064 |

Identical within noise, and min10 trains 3.3× faster. `min_child_samples=20` makes
any column present in fewer than 20 molecules unusable for a tree split. So **rare
substructures carry usable signal for an additive model but not for a tree
ensemble** — which justifies min10 for all LightGBM work.

---

## 5. Interpretation: the main result

### 5.1 Method

1. **Column → substructure.** Each fingerprint column stores its identifier, radius,
   an example molecule and the centre atom. `FindAtomEnvironmentOfRadiusN` +
   `MolFragmentToSmiles(allHsExplicit=True)` recovers the actual environment.
   Explicit H counts matter: an aldehyde's `[O]=[CH]` and a ketone's `[O]=[C]` are
   different identifiers and were indistinguishable without them.
2. **Substructure → functional group.** A SMARTS library of 24 groups (all that
   QM9's C/N/O/F composition permits). **Assignment rule: a column belongs to group
   G if its centre atom is one of G's core atoms** — a Morgan identifier is defined
   by its centre, and the environment is context.
3. **Per-molecule assignment.** Low-radius identifiers are context-free: the
   radius-0 carbonyl oxygen occurs in 4,890 molecules spanning aldehydes, ketones,
   esters and amides. Its group therefore cannot be fixed per column; it is resolved
   **per molecule**, from the atom it is centred on in that molecule.
4. **Aggregate.** For each molecule, sum SHAP over all columns assigned to each
   group. The result is a distribution of per-molecule contributions in eV.

SHAP for Ridge is exact and closed-form, φⱼ = wⱼ(xⱼ − x̄ⱼ), and never requires
densifying the 161k-column matrix — the full test set runs in seconds.

### 5.2 Models used for interpretation

| role | model | test MAE (HOMO/LUMO) | SHAP cost |
|---|---|---|---|
| best predictor (reported) | LightGBM, 11,408 trees × 255 leaves | 0.098 / 0.094 | 1.8 s/molecule — impractical |
| **primary interpretation** | Ridge, CF-MF min1 | 0.118 / 0.143 | seconds for 13k molecules |
| nonlinear cross-check | LightGBM small, 6,000 × 63 | 0.107 / 0.105 | 0.10 s/molecule |
| independent cross-check | RF on 44 `fr_*` counts only | 0.254 / 0.487 | fast |
| descriptor explanation | RF, `min_samples_leaf=10` | 0.132 / 0.142 | 9–14 min |

The `fr_*`-only model is an interpretability probe, not a competitive model — its
poor MAE is expected and reported so it isn't mistaken for one.

### 5.3 Group contributions (eV, median per molecule)

**LUMO** — clean, consistent, chemically correct:

| group | Ridge | LightGBM | RF `fr_*` | frac. negative |
|---|---|---|---|---|
| carbonyl C=O | −1.23 | −1.16 | −1.25 | 0.97 |
| nitrile | −0.90 | −0.77 | −0.52 | 1.00 |
| imine | −0.79 | −0.72 | −0.50 | 1.00 |
| pyridine-type N | −0.76 | −0.39 | — | 1.00 |
| alkene | −0.50 | −0.65 | — | 1.00 |
| alkyne | −0.47 | −0.48 | −0.39 | 0.99 |
| fluoride | −0.29 | — | — | 1.00 |
| pyrrole-type NH | **+0.30** | +0.23 | — | 0.01 |

Every π-acceptor lowers the LUMO. The only group that raises it is the
electron-rich pyrrole NH.

**LUMO carbonyl subtypes** (stable, reported separately):

| subtype | median | frac. negative |
|---|---|---|
| ketone | −1.48 | 1.00 |
| aldehyde | −1.34 | 1.00 |
| ester | −0.73 | 0.88 |
| amide | −0.38 | 0.91 |

The ordering follows donation from the adjacent heteroatom raising the π*.

**HOMO** — correct, but with one group showing no net effect:

| group | Ridge | LightGBM | frac. negative |
|---|---|---|---|
| nitrile | −0.66 | −0.56 | 1.00 |
| pyridine-type N | −0.40 | −0.19 | 1.00 |
| fluoride | −0.21 | — | 1.00 |
| epoxide | −0.13 | −0.10 | 0.85 |
| **carbonyl C=O** | **−0.03** | **−0.04** | **0.60** |
| ether O | +0.04 | +0.06 | 0.40 |
| alkene | +0.09 | +0.19 | 0.06 |
| tertiary amine | +0.24 | +0.18 | 0.08 |
| secondary amine | +0.28 | +0.28 | 0.09 |
| pyrrole-type NH | +0.37 | +0.33 | 0.00 |
| phenol OH | +0.49 | +0.33 | 0.00 |
| primary amine | +0.69 | +0.66 | 0.00 |

### 5.4 The main chemical finding: HOMO/LUMO carbonyl asymmetry

The carbonyl shows a **large, highly consistent LUMO effect** (−1.16 to −1.25 eV,
negative in 97% of molecules) and **no consistent HOMO effect** (−0.03 to −0.04 eV,
negative in only 60%, IQR ≈ 0.25 eV).

Both the additive Ridge model and the nonlinear LightGBM model reproduce this
independently, and the `fr_*`-only RF agrees in sign. Two models with completely
different inductive biases arriving at the same answer rules out an artifact of
either.

**Interpretation.** A carbonyl exerts two opposing influences on occupied orbitals:
the oxygen lone pair n(O) is relatively high-lying and pushes the HOMO up, while the
group's strong inductive withdrawal pulls occupied orbitals down. In QM9's small
molecules these are comparable, so the net effect is near zero with a wide spread.
No such cancellation exists for the LUMO, where the carbonyl π* **is** the LUMO.

Contrast with the nitrile (−0.66 eV on HOMO, consistent): its lone pair is
sp-hybridised and much lower-lying, so there is no upward push to cancel the
withdrawal.

A context test — splitting carbonyl-containing molecules by whether they also
contain an amine, alkene or alcohol — found the near-zero contribution in **all**
contexts (medians −0.05 to −0.01), so this is a genuine cancellation rather than
context-dependence averaging out.

### 5.5 The IQR as a context-dependence measure

The interquartile range of the per-molecule distribution measures how much a
group's effect depends on the rest of the molecule. LUMO:

| context-independent | IQR | context-dependent | IQR |
|---|---|---|---|
| nitrile | 0.18 | carbonyl C=O | 0.87 |
| aldehyde | 0.25 | pyridine-type N | 0.84 |
| imine | 0.30 | alkene | 0.73 |

This is a second axis of information that a standard SHAP importance ranking
discards.

### 5.6 Where the model's behaviour lives

Of 161,375 fingerprint columns, **37 columns carry 55% of the total |SHAP|** —
the radius-0 atom types and the most common radius-1 environments. The 33,069
columns with support ≤10 molecules carry 12% between them.

| support (molecules) | n columns | share of \|SHAP\| |
|---|---|---|
| >1000 | 37 | 55% |
| 101–1000 | 199 | 20% |
| 11–100 | 1,527 | 13% |
| 1–10 | 33,069 | 12% |

This quantifies why individual radius-2 substructures cannot be interpreted alone,
and why group aggregation is the correct unit of analysis.

### 5.7 Descriptor model SHAP

**LUMO is dominated by three descriptors**, with a 6× drop to the fourth:

| descriptor | mean \|SHAP\| | direction |
|---|---|---|
| HallKierAlpha | 0.438 | + |
| FractionCSP3 | 0.242 | + |
| VSA_EState2 | 0.226 | − |

All three encode the same fact: **saturated molecules have high LUMOs, unsaturated
ones low LUMOs**. Consistent with the group analysis, where every π system lowered
the LUMO, and it explains why removing electronic descriptors barely hurt LUMO —
the dominant signal is carried by saturation/shape descriptors.

**HOMO importance is spread evenly** — no family exceeds 20%, top descriptor only
8%. The interpretable standouts are `fr_aniline` (direction +0.96, raises HOMO) and
`fr_nitrile` (−0.98, lowers it), matching the fingerprint result from a different
representation.

---

## 6. Validation

### 6.1 Bootstrap stability

10 Ridge refits on bootstrap resamples of the training set:

| group | mean | sd |
|---|---|---|
| LUMO carbonyl | −1.247 | 0.024 |
| LUMO nitrile | −0.909 | 0.008 |
| HOMO primary amine | +0.680 | 0.008 |
| HOMO nitrile | −0.651 | 0.013 |

**All groups: sd ≤ 0.025 eV.** The attributions are properties of the data, not of
one particular fit.

### 6.2 Removal and sufficiency (LightGBM)

| group | SHAP | removal | agreement | sufficiency r |
|---|---|---|---|---|
| LUMO carbonyl | −1.16 | −1.54 | 1.33 | **0.74** |
| LUMO nitrile | −0.77 | −0.59 | 0.76 | 0.47 |
| HOMO nitrile | −0.56 | −0.61 | 1.10 | 0.42 |
| HOMO epoxide | −0.10 | −0.14 | 1.37 | 0.14 |
| HOMO pyridine N | −0.19 | −0.02 | **0.09** | 0.22 |
| LUMO pyridine N | −0.39 | −0.07 | **0.19** | 0.31 |

- **Carbonyl and nitrile attributions are load-bearing** — removing their columns
  shifts predictions by 76–137% of the attributed value.
- **LUMO carbonyl sufficiency 0.74** is the highest observed: those columns alone
  reproduce 74% of the prediction's deviation. The quantitative form of "the
  carbonyl π* is the LUMO".
- **Aromatic N and the amines fail the removal test** (agreement 0.09–0.52) despite
  having the *highest* sufficiency scores. Their structural information is
  redundantly encoded across many columns, so zeroing one subset changes little.
  These attributions are reported with that caveat.

**General limitation:** zeroing columns produces feature vectors no real molecule
has, so the model extrapolates. Removal magnitudes are upper bounds, and agreement
far from 1.0 may reflect extrapolation rather than a bad attribution.

---

## 7. Known limitations

1. **QM9 contains no Cl or S.** Only C, H, N, O, F. No conclusions about chlorine,
   sulfur, or any group containing them are possible from this dataset.
2. **HOMO carbonyl subtypes could not be separated.** Aldehyde/ketone/ester/amide
   share the radius-0 C=O oxygen identifier; the additive model assigns it a single
   weight and subtype deviations get pushed into carbon-centred features where they
   can invert the shared term. Carbonyls are merged for HOMO. LUMO is unaffected
   because the shared term dominates there.
3. **Scaffold test set excludes acyclic molecules** and is shifted +0.14 eV in HOMO
   relative to train.
4. **LightGBM val MAEs are slightly optimistic** — early stopping chose the tree
   count using the val set. Test numbers are unaffected.
5. **The interpretation models are not the best models.** Ridge is 20%/52% worse
   than full LightGBM; the small LightGBM is 9%/12% worse; the interpretation RF is
   9%/14% worse. Accuracy was traded for tractable SHAP, and every gap is reported.
6. **Group assignment uses the centre atom.** Group totals mean "the summed
   contribution of all environments *centred on* this group's atoms", not everything
   about the group. 55% of columns match more than one group.
7. **`aromatic N` in the `fr_*` cross-check merges pyridine/pyrrole/imidazole**,
   which have opposing effects — its IQR of 0.57–1.16 makes it uninterpretable and
   it is excluded from comparisons.
8. **Literature comparisons are not controlled** — different splits and cleaning.
   They place the result in a class, not in a ranking.

---

## 8. File map

```
config.py                 paths, seed, unit constants
qm9lib/
  data.py                 loaders, parameter storage
  splits.py               random + Bemis-Murcko scaffold splits
  descriptors.py          RDKit descriptors, train-fitted filter
  cfmf.py                 unfolded Morgan fingerprints, column selection
  models.py               fit_rf / fit_ridge / fit_lgbm, evaluate
  fragments.py            column -> substructure extraction
  groups.py               SMARTS library, centre-atom assignment
  explain.py              per-molecule group SHAP aggregation

01_audit.py               read-only inspection of raw QM9
02_clean.py               -> qm9_clean.parquet (130,741)
03_split.py               -> splits.json
04a_descriptors.py        -> desc_raw.parquet, desc_selected.json
04b_cfmf.py               -> morgan_r2_counts.npz, meta, selections
05a_screen.py             8 fingerprint configs via Ridge
05a_followup.py           alpha grid + min1/min10 on both splits
05b_tune_desc.py          RF random search
05c_tune_cfmf.py          LightGBM random search
06a_train.py              12 final models -> results_test.csv
07a_shap_ridge.py         Ridge SHAP + fragment extraction
08_groups.py              columns -> functional groups
09_group_effects.py       per-molecule group totals
10_crosscheck_lgbm.py     LightGBM TreeSHAP group totals
11_figure.py              main group-effect figure
12_crosscheck_desc.py     fr_*-only RF probe
13_validation.py          stability, removal, sufficiency
14_ablations.py           descriptor and fingerprint ablations
15_shap_desc.py           descriptor RF SHAP + beeswarm plots
16_outputs.py             parity figures (3 styles) + all tables
```

---

## 9. Summary of claims supported by this work

1. Collision-free Morgan fingerprints can be constructed directly from unfolded
   identifiers, with no length search — verified zero collisions on 130,741 molecules.
2. A SMILES-only LightGBM model reaches 0.098 / 0.094 eV on QM9 HOMO/LUMO,
   competitive for its model class and ~3.5× above the conformer-noise floor for
   2D representations.
3. Dataset-wide group-level SHAP aggregation yields chemically correct, reproducible
   functional-group contributions, validated across three representations, two model
   families, bootstrap resampling, and removal/sufficiency tests.
4. The carbonyl group shows a strong consistent LUMO effect and no net HOMO effect,
   reproduced independently by additive and nonlinear models — attributable to
   competing lone-pair and inductive influences on occupied orbitals.
5. Rare substructures carry usable signal for additive models but not for tree
   ensembles, due to the minimum-samples-per-leaf constraint.
6. The most accurate model degrades most under scaffold splitting, and random-split
   QM9 scores are optimistic because the dataset is rich in near-twin molecules.